# Assignment 3: Predict atmospheric neutral density using GRACE satellite data

**Goal:** predict the GRACE **neutral density** (`density`) using the GRACE orbit variables + OMNI space weather variables, with **Gaussian Process Regression (GPR)**.


## 1) Imports + scoring function



In [22]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel

from sklearn.metrics import mean_squared_error


# Scoring function

def print_metrics(y, y_pred):
    # 1. RMSE
    rmse = np.sqrt(mean_squared_error(np.log(y), np.log(y_pred)))
    print(f"RMSE: {rmse:.2f}")

    # 2. R
    R = np.corrcoef(np.log(y), np.log(y_pred))[0,1]
    print(f"Corr coeff: {R:.2f}")

    score = 100*np.min((R/rmse/0.74,1))
    print(f"Score =  {score:.2f}")

    return score

RANDOM_STATE = 42
rng = np.random.RandomState(RANDOM_STATE)


## 2) Load **training** data only


In [23]:
train_path = "grace_sat1_hourly_train.csv"
train_raw = pd.read_csv(train_path)

print("Train shape:", train_raw.shape)
train_raw.head()


Train shape: (111936, 16)


,time,altitude,longitude,latitude,local_solar_time,density,validity_flag,Bx,By,Bz,proton_density,temperature,V,E,beta,Dst
0,04-Apr-2002 00:00:00,504941.967921,-22.181309,66.409713,22.468284,9.990000e+32,1,1.2,-0.3,1.4,2.7,130922,539,-0.75,3.46,-2
1,04-Apr-2002 01:00:00,511261.746533,144.674401,-17.774022,10.592202,1.634391e-12,0,-0.1,2.0,-0.2,3.0,125714,523,0.10,3.52,-1
2,04-Apr-2002 02:00:00,492768.204375,-49.478123,-31.054624,22.648903,8.110523e-13,0,-1.3,2.0,-0.1,2.7,130563,511,0.05,2.84,-3
3,04-Apr-2002 03:00:00,514088.178295,119.883059,78.999367,10.939851,1.477093e-12,0,-2.5,0.6,0.8,2.5,152417,503,-0.40,2.53,-5
4,04-Apr-2002 04:00:00,518381.771002,98.584507,-52.417285,10.520151,1.271249e-12,0,-1.7,1.6,-0.7,2.6,119842,503,0.35,2.32,-7


## 3) Cleaning utilities

In [31]:
# Fill value definitions from the assignment table

FILL_VALUES = {
    "Bx": 999.9,
    "By": 999.9,
    "Bz": 999.9,
    "temperature": 9999999,
    "proton_density": 999.9,
    "V": 9999,
    "E": 999.99,
    "beta": 999.99,
    "Dst": 99999,
}

def replace_fill_values(df: pd.DataFrame) -> pd.DataFrame:
    """Replace OMNI/GRACE fill values (and infs) with NaN."""
    df = df.copy()

    # OMNI fill values
    for col, fill in FILL_VALUES.items():
        if col in df.columns:
            df.loc[df[col] == fill, col] = np.nan

    # Some GRACE columns contain a huge fill value (commonly ~9.99e32)
    for col in ["altitude", "longitude", "latitude", "local_solar_time"]:
        if col in df.columns:
            df.loc[df[col] > 1e20, col] = np.nan

    # Convert non-finite density to NaN
    if "density" in df.columns:
        df.loc[~np.isfinite(df["density"]), "density"] = np.nan

    return df


def clean_training(df: pd.DataFrame) -> pd.DataFrame:
    """Training-only cleaning: remove invalid rows."""
    df = replace_fill_values(df)

    # 1) Remove anomalous GRACE data
    df = df[df["validity_flag"] == 0].copy()

    # 2) Basic physical sanity checks (set bad values to NaN)
    df.loc[(df["altitude"] < 200e3) | (df["altitude"] > 1000e3), "altitude"] = np.nan
    df.loc[(df["longitude"] < -180) | (df["longitude"] > 180), "longitude"] = np.nan
    df.loc[(df["latitude"] < -90) | (df["latitude"] > 90), "latitude"] = np.nan
    df.loc[(df["local_solar_time"] < 0) | (df["local_solar_time"] > 24), "local_solar_time"] = np.nan

    # OMNI: basic validity
    df.loc[df["temperature"] <= 0, "temperature"] = np.nan
    df.loc[df["V"] <= 0, "V"] = np.nan
    df.loc[df["proton_density"] <= 0, "proton_density"] = np.nan
    df.loc[df["beta"] < 0, "beta"] = np.nan

    # Target must be positive
    df.loc[df["density"] <= 0, "density"] = np.nan

    # 3) Drop any row with NaN (training-only)
    df = df.dropna(axis=0).reset_index(drop=True)

    return df


train = clean_training(train_raw)

print("After cleaning:")
print("  cleaned train shape:", train.shape)
print("  removed rows:", len(train_raw) - len(train))


After cleaning:
  cleaned train shape: (108880, 16)
  removed rows: 3056


## 4) Feature engineering

### Why feature engineering helps

- Time and longitude are **periodic** (wrap around), so we encode them using `sin`/`cos`.
- We create simple time descriptors: day-of-year and hour-of-day (both periodic).
- We also keep OMNI solar wind / geomagnetic indices as continuous inputs.

We keep the target as **log-density**:  
\[
y = \log(\rho)
\]
This makes the regression numerically stable and closer to a Gaussian distribution (better for GPs).


In [32]:
def make_features(df: pd.DataFrame, reference_time: pd.Timestamp | None = None):
    """Create model features from the raw dataframe.

    Returns:
        X (pd.DataFrame): engineered features
        reference_time (pd.Timestamp): used so train/test share the same time origin
    """
    df = df.copy()

    # Parse timestamps
    dt = pd.to_datetime(df["time"], format="%d-%b-%Y %H:%M:%S", errors="coerce")

    # Choose a time origin from the *training* data
    if reference_time is None:
        reference_time = dt.min()

    days_since_start = (dt - reference_time).dt.total_seconds() / 86400.0
    day_of_year = dt.dt.dayofyear.astype(float)
    hour = dt.dt.hour.astype(float)

    # Cyclic encodings
    doy_rad = 2 * np.pi * (day_of_year / 365.25)
    hour_rad = 2 * np.pi * (hour / 24.0)
    lst_rad = 2 * np.pi * (df["local_solar_time"].astype(float) / 24.0)
    lon_rad = np.deg2rad(df["longitude"].astype(float))

    X = pd.DataFrame(
        {
            # Orbit / location
            "alt_km": df["altitude"].astype(float) / 1000.0,
            "lat_deg": df["latitude"].astype(float),
            "lon_sin": np.sin(lon_rad),
            "lon_cos": np.cos(lon_rad),
            "lst_sin": np.sin(lst_rad),
            "lst_cos": np.cos(lst_rad),

            # Time (periodic + slow drift)
            "doy_sin": np.sin(doy_rad),
            "doy_cos": np.cos(doy_rad),
            "hour_sin": np.sin(hour_rad),
            "hour_cos": np.cos(hour_rad),
            "days_since_start": days_since_start.astype(float),

            # OMNI variables
            "Bx": df["Bx"].astype(float),
            "By": df["By"].astype(float),
            "Bz": df["Bz"].astype(float),
            "Np": df["proton_density"].astype(float),
            "Tp": df["temperature"].astype(float),
            "V": df["V"].astype(float),
            "E": df["E"].astype(float),
            "beta": df["beta"].astype(float),
            "Dst": df["Dst"].astype(float),
        }
    )

    # A useful derived feature: magnetic field magnitude
    X["Bmag"] = np.sqrt(X["Bx"]**2 + X["By"]**2 + X["Bz"]**2)

    return X, reference_time


X_train, ref_time = make_features(train)
y_train_log = np.log(train["density"].values)

print("Feature matrix:", X_train.shape)
print("Target shape:", y_train_log.shape)


Feature matrix: (108880, 21)
Target shape: (108880,)


## 5) Preprocessing

Gaussian Processes use distance in feature space, so **scaling** matters.

We use a standard preprocessing pipeline:

1) `SimpleImputer(median)`  
   - Just in case any value becomes NaN (especially useful for test-time robustness).

2) `StandardScaler()`  
   - Makes each feature roughly mean 0, std 1, so a single kernel length-scale is meaningful.


In [33]:
preprocess = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

X_train_scaled = preprocess.fit_transform(X_train)
print("Scaled shape:", X_train_scaled.shape)


Scaled shape: (108880, 21)


## 6) Split the dataset into subsets (PCA + GMM clustering)


- **GMM** clusters data by fitting a mixture of Gaussians (soft clustering).
- In higher dimensions, GMM can be slow; PCA reduces dimensionality for faster clustering.
- We only use PCA/GMM to decide **which GP expert** handles which region of feature space.

We cluster on a **small sample** (for speed) and then label all training points.


In [34]:
# Clustering hyperparameters

N_CLUSTERS = 10
PCA_COMPONENTS = 10
CLUSTER_FIT_SAMPLE = 20000  # fit PCA and GMM on a subset for speed

# 1) Fit PCA on a subset
fit_idx = rng.choice(len(X_train_scaled), size=min(CLUSTER_FIT_SAMPLE, len(X_train_scaled)), replace=False)
pca = PCA(n_components=PCA_COMPONENTS, svd_solver="randomized", random_state=RANDOM_STATE)
X_fit_pca = pca.fit_transform(X_train_scaled[fit_idx])

# 2) Fit GMM in PCA space
gmm = GaussianMixture(
    n_components=N_CLUSTERS,
    covariance_type="diag",
    random_state=RANDOM_STATE,
    max_iter=200,
    reg_covar=1e-6,
    n_init=1,
)
gmm.fit(X_fit_pca)

# 3) Label all training data
X_train_pca = pca.transform(X_train_scaled)
train_cluster = gmm.predict(X_train_pca)

cluster_sizes = np.bincount(train_cluster, minlength=N_CLUSTERS)
print("Cluster sizes:", cluster_sizes)


Cluster sizes: [12763 10398 10401 13898 10961 11886 13721  4759  8394 11699]


## 7) Train Gaussian Process experts

Hyperparameter optimization of a GP kernel is expensive.  
A practical trick:

1) Fit **one pilot GP** on a random sample (e.g., 800 points) *with optimization* → learn sensible kernel parameters.
2) Freeze the learned kernel and train each cluster GP with `optimizer=None` (fast).

This keeps everything “GP-based” while making it computationally feasible.


In [35]:
# 7.1 Pilot GP

PILOT_POINTS = 800

pilot_idx = rng.choice(len(X_train_scaled), size=min(PILOT_POINTS, len(X_train_scaled)), replace=False)

pilot_kernel = C(1.0, (1e-2, 1e2)) * RBF(length_scale=1.0, length_scale_bounds=(1e-2, 1e2)) + WhiteKernel(
    noise_level=1e-2, noise_level_bounds=(1e-6, 1e0)
)

pilot_gp = GaussianProcessRegressor(
    kernel=pilot_kernel,
    normalize_y=True,
    random_state=RANDOM_STATE,
    n_restarts_optimizer=0,
)

pilot_gp.fit(X_train_scaled[pilot_idx], y_train_log[pilot_idx])
tuned_kernel = pilot_gp.kernel_  # learned hyperparameters

print("Tuned kernel:", tuned_kernel)


Tuned kernel: 1.1**2 * RBF(length_scale=3.57) + WhiteKernel(noise_level=0.0715)


In [51]:
# 7.2 Train one GP per cluster 

POINTS_PER_CLUSTER = 750 
MIN_POINTS_FOR_CLUSTER = 200

cluster_gps = {}
for k in range(N_CLUSTERS):
    idx_k = np.where(train_cluster == k)[0]

    if len(idx_k) < MIN_POINTS_FOR_CLUSTER:
        cluster_gps[k] = None
        print(f"Cluster {k}: too small ({len(idx_k)}). Will use fallback GP.")
        continue

    train_idx_k = rng.choice(idx_k, size=min(POINTS_PER_CLUSTER, len(idx_k)), replace=False)

    gp_k = GaussianProcessRegressor(
        kernel=tuned_kernel,
        optimizer=None,          # freeze kernel parameters for speed
        normalize_y=True,
        random_state=RANDOM_STATE,
    )
    gp_k.fit(X_train_scaled[train_idx_k], y_train_log[train_idx_k])
    cluster_gps[k] = gp_k
    print(f"Cluster {k}: trained on {len(train_idx_k)} points")

# Fallback GP (used if a cluster GP is missing)
fallback_gp = GaussianProcessRegressor(
    kernel=tuned_kernel,
    optimizer=None,
    normalize_y=True,
    random_state=RANDOM_STATE,
)
fallback_gp.fit(X_train_scaled[pilot_idx], y_train_log[pilot_idx])

# Useful clipping range (prevents extreme exp() blow-ups)
log_lo, log_hi = np.percentile(y_train_log, [0.5, 99.5])


Cluster 0: trained on 750 points
Cluster 1: trained on 750 points
Cluster 2: trained on 750 points
Cluster 3: trained on 750 points
Cluster 4: trained on 750 points
Cluster 5: trained on 750 points
Cluster 6: trained on 750 points
Cluster 7: trained on 750 points
Cluster 8: trained on 750 points
Cluster 9: trained on 750 points


## 8) Predict on the **test set** and report metrics


In [52]:
test_path = "grace_sat1_hourly_test.csv"
test_raw = pd.read_csv(test_path)

# IMPORTANT: do NOT drop rows from the test set (assignment requirement)
test_proc = replace_fill_values(test_raw)

# If anything is unphysical, we mark as NaN so the imputer can handle it
test_proc.loc[(test_proc["altitude"] < 200e3) | (test_proc["altitude"] > 1000e3), "altitude"] = np.nan
test_proc.loc[(test_proc["longitude"] < -180) | (test_proc["longitude"] > 180), "longitude"] = np.nan
test_proc.loc[(test_proc["latitude"] < -90) | (test_proc["latitude"] > 90), "latitude"] = np.nan
test_proc.loc[(test_proc["local_solar_time"] < 0) | (test_proc["local_solar_time"] > 24), "local_solar_time"] = np.nan
test_proc.loc[test_proc["temperature"] <= 0, "temperature"] = np.nan
test_proc.loc[test_proc["V"] <= 0, "V"] = np.nan
test_proc.loc[test_proc["proton_density"] <= 0, "proton_density"] = np.nan
test_proc.loc[test_proc["beta"] < 0, "beta"] = np.nan

# Features + preprocessing
X_test, _ = make_features(test_proc, reference_time=ref_time)
X_test_scaled = preprocess.transform(X_test)

# Cluster assignment in PCA space
X_test_pca = pca.transform(X_test_scaled)
test_cluster = gmm.predict(X_test_pca)

# Predict log-density
y_pred_log = np.empty(len(X_test_scaled))
for k in range(N_CLUSTERS):
    mask = (test_cluster == k)
    if not np.any(mask):
        continue

    model = cluster_gps.get(k)
    if model is None:
        y_pred_log[mask] = fallback_gp.predict(X_test_scaled[mask])
    else:
        y_pred_log[mask] = model.predict(X_test_scaled[mask])

# Clip + back-transform to physical density
y_pred_log = np.clip(y_pred_log, log_lo - 1.0, log_hi + 1.0)
pred = np.exp(y_pred_log)

# Ground truth (provided in test CSV)
test = test_raw["density"].values

print_metrics(test, pred)


RMSE: 0.55
Corr coeff: 0.92
Score =  100.00


np.float64(100.0)